In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
import time
import warnings
from pathlib import Path
from typing import Any
from datasets import load_dataset

In [ ]:
HUB_DATASET = "DenCT/laya-decisions-intake-10k"

TRAIN_FILE = "intake_v1.train.jsonl"
EVAL_FILE = "intake_v1.eval.jsonl"
DEFAULT_BASE = "multilingual"
DEFAULT_OUT = "laya-intake-v1"

BASELINE_NOTE = """Stock-checkpoint baseline on the section 6.1 catalogue (measured 2026-10-14):
performance_status -> ecog_0      @ 0.836  WRONG (truth ECOG 1), clears any gate
age_band           -> under_18    @ 0.297  wrong (truth 63), caught by abstention
smoking_status     -> former      @ 0.634  right, discarded by the 0.65 gate
prior_lines        -> 1           @ 0.50   wrong (truth 3), caught by abstention
sex                -> female      @ 0.908  right
menopausal         -> postmeno.   @ 0.864  right
*_stated (noul)    -> collapsed; stated and unstated are indistinguishable
"""

# Training JSONL filepath.
train_data = "intake_v1.train.jsonl"
# Eval JSONL, used for the before/after comparison. 
eval_data = "intake_v1.eval.jsonl"
# Skip the held-out eval split; metrics then come from the in-sample calibration slice only.
no_eval = False
# Base checkpoint: a local directory, or a name laya resolves and downloads from the Hub.
base = "multilingual"
# Output directory for the fine-tuned checkpoint. 
out = "output"
# Build every training item WITHOUT loading model weights, print the stats, and stop.
dry_run = False
# Do not run the dry-run check before training.
skip_dry_run = True
# Train even though the dry run dropped items. A wrong label literal does not raise -- laya silently drops 
# the item -- so by default any non-empty 'skipped' aborts before the GPU is touched.
allow_skipped = True

# Hyperparameters
# laya's own default is 4; 6 is this project's default because a ~10k-item corpus is small enough for
# the head to collapse to the class prior.
epochs = 6
# Re-encode choice questions with a random option order each epoch, so the model cannot learn a position prior.
# laya's default is off; on here because the measured failure is a 6-option ECOG head.
shuffle_options = "off" # "on"
# Rows per forward pass.
micro_batch = 8
# Accumulation steps; micro-batch x this is the effective batch.
grad_accum = 8
encoder_lr = 2.5e-5
head_lr = 1e-4
loss = "rlcd" # "soft-ce"
label_smoothing = 0.0
# Train the heads only. Fits in far less memory, but leaves ECOG inference -- an encoder-level task -- untouched (TRD R10).
freeze_encoder = True
seed = 42

# Token budgets and precision
max_len = 1024
# Question-head token budget. Defaults to the checkpoint's (256).
head_max_len = 256
# Mixed precision. 'auto' follows the device: on for CUDA, off otherwise. Try 'off' first if the loss goes NaN.
amp = "auto" # "on", "off"
# Trade compute for VRAM in the encoder and head. 'auto' follows whatever AMP resolved to.
gradient_checkpointing = "auto" # "on", "off"

# Calibration and abstention
# Tolerated error among accepted answers when fitting the per-bucket abstention thresholds.
target_error = 0.10
# Minimum calibration items in a bucket before a threshold is fitted.
min_abstain_n = 10
device = "cuda"

# Progress reporting
# Replace laya's per-step log lines with a tqdm bar (epoch, ETA, raw and smoothed loss).
# "off" keeps laya's plain lines. Either way the lines still reach train.log.
progress = "auto" # "off"
# Micro-batches between laya's loss lines, which is also how often the bar moves.
# None -> 10 with the bar, laya's own 100 without.
log_every = None

In [ ]:
def _disable_broken_truststore() -> None:
    """Undo truststore's SSL injection on Python 3.14, where it recurses forever."""
    if sys.version_info < (3, 14):
        return
    import ssl

    if "truststore" not in getattr(ssl.SSLContext, "__module__", ""):
        return
    for name in ("pip._vendor.truststore", "truststore"):
        try:
            module = __import__(name, fromlist=["extract_from_ssl"])
            module.extract_from_ssl()
        except Exception:  # noqa: BLE001 - try the next candidate module
            continue
        if "truststore" not in getattr(ssl.SSLContext, "__module__", ""):
            print(
                "note: removed truststore's SSL injection (%s); certificates now come "
                "from certifi." % name,
                flush=True,
            )
            return


def _tristate(value: str) -> bool | None:
    """'auto' -> None, which is how TrainConfig spells 'follow the device'."""
    return {"auto": None, "on": True, "off": False}[value]


def _find_input(explicit: str | None, filename: str, what: str) -> str:
    """Resolve a corpus path: the flag, then beside this script, then the repo."""
    if explicit:
        path = Path(explicit).expanduser()
        if not path.exists():
            sys.exit("train_laya: %s file not found: %s" % (what, path))
        return str(path)

    candidates = [
        SCRIPT_DIR / filename,
        Path.cwd() / filename,
        SYNTHETIC_DIR / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    sys.exit(
        "train_laya: could not find the %s corpus (%s). Looked in:\n  %s\n"
        "Pass an explicit path, or upload the file next to this script."
        % (what, filename, "\n  ".join(str(c) for c in candidates))
    )


# --- corpus download ---------------------------------------------------------
# Kaggle's writable output is /kaggle/working: what lands there shows up under
# the notebook's Output tab and is what you download when the session ends.
# Anywhere else (Colab, a laptop) fall back to the working directory.
KAGGLE_WORKING = Path("/kaggle/working")
WORK_DIR = KAGGLE_WORKING if KAGGLE_WORKING.is_dir() else Path.cwd()
# A notebook has no __file__; point the script-era lookups at the same place.
SCRIPT_DIR = SYNTHETIC_DIR = WORK_DIR

# Hub split -> the filename the rest of the notebook expects.
HUB_SPLITS = {"train": TRAIN_FILE, "validation": EVAL_FILE}
# Key order of backend/data/synthetic/intake_v1.*.jsonl, reproduced so the
# downloaded corpus diffs cleanly against the committed one.
CORPUS_KEYS = (
    "state",
    "questions",
    "expected",
    "gold",  # soft-label probabilities; present on ~29% of rows, omitted elsewhere
    "seed",
    "generator",
    "domain",
    "presentation",
)
# Re-download even when the JSONL is already sitting in the working directory.
REDOWNLOAD = False


def _corpus_row(row: dict[str, Any]) -> dict[str, Any]:
    """One Hub row in the synthetic corpus's key order.

    A parquet row carries every column, so an absent `gold` arrives as None;
    the synthetic corpus omits the key instead, and so does this.
    """
    extras = [key for key in row if key not in CORPUS_KEYS]
    return {
        key: row[key] for key in (*CORPUS_KEYS, *extras) if row.get(key) is not None
    }


def download_corpus(
    repo_id: str = HUB_DATASET,
    dest: str | Path | None = None,
    revision: str | None = None,
    overwrite: bool = False,
) -> dict[str, Path]:
    """Download the intake corpus from the Hub as intake_v1.{train,eval}.jsonl.

    The Hub copy is backend/data/synthetic re-serialised as parquet: same
    columns, same 3071/342 rows, so what lands here is the format laya's
    loader already consumes. It is not byte-identical -- parquet normalises a
    few soft-label floats (0.39999999999999997 -> 0.4) -- but it parses equal.
    An existing file is reused rather than refetched, so re-running the cell
    is free; pass overwrite=True (REDOWNLOAD) to force a refresh.
    """
    _disable_broken_truststore()
    dest_dir = Path(dest).expanduser() if dest else WORK_DIR
    dest_dir.mkdir(parents=True, exist_ok=True)

    written: dict[str, Path] = {}
    for split, filename in HUB_SPLITS.items():
        path = dest_dir / filename
        if path.exists() and not overwrite:
            print(
                "reusing %s (%.1f KB) -- set REDOWNLOAD = True to refetch"
                % (path, path.stat().st_size / 1024),
                flush=True,
            )
            written[split] = path
            continue

        print("downloading %s [%s] ..." % (repo_id, split), flush=True)
        try:
            dataset = load_dataset(repo_id, split=split, revision=revision)
        except Exception as err:  # noqa: BLE001 - datasets raises many types here
            sys.exit(
                "train_laya: could not load %s [%s]: %s\n"
                "  On Kaggle this is usually the session's Internet toggle: open the\n"
                "  right-hand panel, Notebook options -> Internet on (needs a verified\n"
                "  phone number). A gated or private dataset needs an HF_TOKEN secret."
                % (repo_id, split, err)
            )

        # Write through a temp file: an interrupted session cannot leave a half
        # corpus where the trainer looks.
        tmp = path.parent / (path.name + ".tmp")
        rows = 0
        with tmp.open("w", encoding="utf-8", newline="\n") as handle:
            for row in dataset:
                handle.write(json.dumps(_corpus_row(row), ensure_ascii=False) + "\n")
                rows += 1
        os.replace(tmp, path)

        written[split] = path
        print(
            "  %-24s %6d rows, %8.1f KB"
            % (path.name, rows, path.stat().st_size / 1024),
            flush=True,
        )
    return written


corpus = download_corpus(HUB_DATASET, dest=WORK_DIR, overwrite=REDOWNLOAD)
# Point the cell-1 settings at what actually landed, so main() does not have to
# guess the working directory.
train_data = str(corpus["train"])
eval_data = str(corpus["validation"])
print("\nwork dir   %s\ntrain_data %s\neval_data  %s" % (WORK_DIR, train_data, eval_data))

In [ ]:
class _Tee:
    """Mirror a stream to a file, so a closed browser tab does not lose the run."""

    def __init__(self, stream, handle):
        self._stream = stream
        self._handle = handle

    def write(self, text: str) -> int:
        self._stream.write(text)
        self._handle.write(text)
        self._handle.flush()
        return len(text)

    def flush(self) -> None:
        self._stream.flush()
        self._handle.flush()

    def isatty(self) -> bool:
        return self._stream.isatty()


# laya's two training-loop log lines, which the progress bar consumes.
_STEP_LINE = re.compile(r"^epoch (\d+)/(\d+) step (\d+) loss ([\d.]+)$")
_EPOCH_LINE = re.compile(r"^epoch (\d+)/(\d+) mean loss ([\d.]+)$")


class _ProgressBar:
    """Render laya's per-step log lines as a tqdm bar.

    laya hardcodes `print("epoch %d/%d step %d loss %.4f")` in its training
    loop and exposes no progress hook, so there is nothing to subscribe to.
    What there is: `print` inside laya/train.py resolves as a module global
    before it reaches builtins, so binding one there reroutes those lines
    without patching the installed package -- which matters, because on Kaggle
    laya is a fresh pip install that no local edit would survive.
    """

    def __init__(self, tqdm_cls, epochs: int, steps_per_epoch: int | None, stream, log_handle=None):
        self._epochs = epochs
        self._steps_per_epoch = steps_per_epoch
        self._log = log_handle
        self._epoch = 1
        self._step = 0  # last step number seen inside the current epoch
        self._ema: float | None = None
        self._bar = tqdm_cls(
            total=epochs * steps_per_epoch if steps_per_epoch else None,
            unit="step",
            dynamic_ncols=True,
            smoothing=0.05,
            file=stream,
            desc="epoch 1/%d" % epochs,
        )

    def handle(self, text: str) -> bool:
        """Absorb a step line into the bar. True when the line was consumed."""
        match = _STEP_LINE.match(text)
        if match:
            self._advance(int(match.group(1)), int(match.group(3)), float(match.group(4)))
            if self._log is not None:  # the per-step trace still reaches train.log
                self._log.write(text + "\n")
            return True

        match = _EPOCH_LINE.match(text)
        if match:
            self._end_epoch(int(match.group(1)))
        return False  # mean-loss lines are the ones worth reading; let them print

    def _advance(self, epoch: int, step: int, loss: float) -> None:
        if epoch != self._epoch:  # laya restarts `step` at 1 every epoch
            self._epoch, self._step = epoch, 0
            self._bar.set_description("epoch %d/%d" % (epoch, self._epochs))
        delta, self._step = max(0, step - self._step), step
        # Raw per-micro-batch loss swings hard on 8 items; show it smoothed too.
        self._ema = loss if self._ema is None else 0.9 * self._ema + 0.1 * loss
        self._bar.set_postfix(loss="%.4f" % loss, ema="%.4f" % self._ema, refresh=False)
        self._bar.update(delta)

    def _end_epoch(self, epoch: int) -> None:
        # Without a dry run the steps-per-epoch is unknown until one finishes;
        # adopt the count so the bar gets a real total from epoch 2 onwards.
        if not self._steps_per_epoch and self._step:
            self._steps_per_epoch = self._step
            self._bar.total = self._epochs * self._steps_per_epoch
        if self._steps_per_epoch:  # resync: log_every rarely divides the epoch
            self._bar.n = min(epoch * self._steps_per_epoch, self._bar.total)
            self._bar.refresh()

    def close(self) -> None:
        self._bar.close()


def _install_progress_bar(epochs: int, steps_per_epoch: int | None, stream, log_handle=None):
    """Point laya.train's `print` at a tqdm bar. Returns a restore callable."""
    try:
        from tqdm.auto import tqdm
    except ImportError:
        print(
            "note: tqdm is not installed, so laya's plain step lines stay as they are.",
            flush=True,
        )
        return None

    import laya.train as laya_train

    bar = _ProgressBar(tqdm, epochs, steps_per_epoch, stream, log_handle)
    had_print = "print" in vars(laya_train)
    original = vars(laya_train).get("print", print)

    def hook(*args, **kwargs):
        if "file" not in kwargs:  # never swallow anything aimed at stderr
            text = " ".join(str(a) for a in args)
            if bar.handle(text):
                return
        original(*args, **kwargs)

    laya_train.print = hook

    def restore() -> None:
        bar.close()
        if had_print:
            laya_train.print = original
        else:
            del laya_train.print

    return restore

In [ ]:
def _describe_gpu() -> str:
    import torch

    index = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(index)
    return "cuda:%d %s, %.1f GB VRAM, capability %d.%d" % (
        index,
        props.name,
        props.total_memory / 1024**3,
        props.major,
        props.minor,
    )


def _require_device(device: str) -> None:
    """Refuse to start a multi-hour run on the wrong device, with a usable message."""
    import torch

    if device.startswith("cuda"):
        if not torch.cuda.is_available():
            sys.exit(
                "train_laya: --device %s was requested but no CUDA device is visible.\n"
                "  torch %s\n"
                "  This is the expected failure on the TrialMatch dev box, which has a\n"
                "  CPU-only torch build and no CUDA GPU (TRD R10). Run this on a GPU\n"
                "  runtime -- see README.md -- or pass --device cpu to train anyway,\n"
                "  which will take many hours and may not fit in memory."
                % (device, torch.__version__)
            )
        print("device: %s" % _describe_gpu(), flush=True)
        print(
            "        AMP (fp16 autocast + GradScaler) and gradient checkpointing resolve to\n"
            "        on for CUDA; override with --amp off / --gradient-checkpointing off.",
            flush=True,
        )
        return

    if device == "cpu":
        print(
            "device: cpu -- WARNING. A full fine-tune of the ~322M-parameter checkpoint\n"
            "        needs roughly 5.2 GB of optimiser state alone (TRD R10). Expect many\n"
            "        hours, and consider --freeze-encoder, which fits in less memory but\n"
            "        trains only the heads.",
            flush=True,
        )
        return

    print("device: %s (unverified)" % device, flush=True)


def _print_by_type(report: dict[str, Any]) -> None:
    """Per-question-type accuracy before and after."""
    before = (report.get("before") or {}).get("by_type") or {}
    after = (report.get("after") or {}).get("by_type") or {}
    names = [n for n in ("choice", "score", "noul") if n in before or n in after]
    if not names:
        return

    print("\n=== Accuracy by question type ===", flush=True)
    print(
        "%-10s %-8s %-10s %-10s %-10s" % ("Type", "Items", "Before", "After", "Delta"),
        flush=True,
    )
    print("-" * 52, flush=True)
    for name in names:
        b = before.get(name) or {}
        a = after.get(name) or {}
        b_acc, a_acc = b.get("accuracy"), a.get("accuracy")
        delta = (
            ("%+.4f" % (a_acc - b_acc))
            if (b_acc is not None and a_acc is not None)
            else "N/A"
        )
        print(
            "%-10s %-8s %-10s %-10s %-10s"
            % (
                name,
                a.get("items", b.get("items", "-")),
                "%.4f" % b_acc if b_acc is not None else "N/A",
                "%.4f" % a_acc if a_acc is not None else "N/A",
                delta,
            ),
            flush=True,
        )
    print("-" * 52, flush=True)
    if "noul" in after:
        print(
            "noul is the '*_stated' head, which was collapsed and unusable on the stock\n"
            "checkpoint. Treat any accuracy near 0.5 here as still collapsed.",
            flush=True,
        )


def _print_env_block(summary: dict[str, Any], out_dir: Path) -> None:
    """The lines to paste into backend/.env. Printed, never written (by decision)."""
    name = out_dir.name
    thresholds = summary.get("abstention_thresholds") or {}

    print(
        "\n=== Paste into backend/.env when you bring the artifacts back ===",
        flush=True,
    )
    print('DECISION_CHECKPOINT_ID="./models/%s"' % name, flush=True)
    print('DECISION_ONNX_PATH="./models/%s-onnx/model.onnx"' % name, flush=True)
    if thresholds:
        compact = json.dumps(
            {k: round(float(v), 4) for k, v in sorted(thresholds.items())},
            separators=(",", ":"),
        )
        print("DECISION_MIN_CONFIDENCE='%s'" % compact, flush=True)
        print(
            "\n  These thresholds are FITTED at --target-error, one cut per option-count\n"
            "  bucket. The values currently in .env are guesses.",
            flush=True,
        )
    else:
        print(
            "\n  No abstention thresholds were fitted: every bucket had fewer than\n"
            "  --min-abstain-n calibration items. Leave DECISION_MIN_CONFIDENCE as it is;\n"
            "  an empty map is refused by the validator.",
            flush=True,
        )

    temperature = summary.get("temperature")
    if temperature:
        print(
            "\nFitted temperature (written into the checkpoint config): %s"
            % [round(float(t), 4) for t in temperature],
            flush=True,
        )

In [ ]:
def main() -> int:
    started = time.time()
    _disable_broken_truststore()

    data_path = _find_input(train_data, TRAIN_FILE, "training")
    eval_path = None if no_eval else _find_input(eval_data, EVAL_FILE, "eval")
    out_dir = Path(out).expanduser() if out else SCRIPT_DIR / DEFAULT_OUT

    try:
        import torch
        # NB: imported under another name. `from laya.train import dry_run` would
        # bind dry_run as a local of main(), shadowing the cell-2 flag for the
        # whole function -- and a function object is always truthy, so the run
        # would take the dry-run branch and return before training.
        from laya.train import TrainConfig, finetune
        from laya.train import dry_run as laya_dry_run
    except ImportError as err:
        sys.exit(
            "train_laya: missing dependency (%s). Install requirements.txt first." % err
        )

    if not dry_run:
        _require_device(device)
        out_dir.mkdir(parents=True, exist_ok=True)

    config = TrainConfig(
        epochs=epochs,
        micro_batch=micro_batch,
        grad_accum=grad_accum,
        encoder_lr=encoder_lr,
        head_lr=head_lr,
        loss=loss,
        label_smoothing=label_smoothing,
        shuffle_options=("choice",) if shuffle_options == "on" else (),
        freeze_encoder=freeze_encoder,
        seed=seed,
        max_len=max_len,
        head_max_len=head_max_len,
        amp=_tristate(amp),
        gradient_checkpointing=_tristate(gradient_checkpointing),
        eval_data=eval_path,
        target_error=target_error,
        min_abstain_n=min_abstain_n,
        # A bar that only moves every 100 micro-batches looks stuck.
        log_every=log_every if log_every is not None else (10 if progress != "off" else 100),
    )
    config.validate()

    if dry_run:
        return _run_dry(laya_dry_run, data_path, base, config, gate=False)[0]

    log_path = out_dir / "train.log"
    handle = log_path.open("a", encoding="utf-8")
    real_stdout, real_stderr = sys.stdout, sys.stderr
    sys.stdout, sys.stderr = _Tee(real_stdout, handle), _Tee(real_stderr, handle)
    warnings.simplefilter("always", RuntimeWarning)

    try:
        print("=" * 72, flush=True)
        print("laya fine-tune  %s" % time.strftime("%Y-%m-%d %H:%M:%S"), flush=True)
        print("  torch        %s" % torch.__version__, flush=True)
        print("  train data   %s" % data_path, flush=True)
        print("  eval data    %s" % (eval_path or "(none)"), flush=True)
        print("  base         %s" % base, flush=True)
        print("  out          %s" % out_dir, flush=True)
        print(
            "  epochs %d, micro-batch %d, grad-accum %d (effective %d), loss %s, "
            "shuffle-options %s"
            % (
                config.epochs,
                config.micro_batch,
                config.grad_accum,
                config.micro_batch * config.grad_accum,
                config.loss,
                config.shuffle_options or "off",
            ),
            flush=True,
        )
        print("=" * 72, flush=True)

        steps_per_epoch = None
        if not skip_dry_run:
            print("\n--- acceptance gate (no weights loaded) ---", flush=True)
            code, dry_summary = _run_dry(
                laya_dry_run, data_path, base, config, gate=not allow_skipped
            )
            if code != 0:
                return code
            train_items = dry_summary.get("train_items")
            if train_items:  # ceil: laya's last micro-batch is a short one
                steps_per_epoch = -(-train_items // config.micro_batch)
            print("--- gate passed ---\n", flush=True)

        restore_print = None
        if progress != "off":
            # The bar goes to the real stderr, never through the _Tee: a
            # redrawn bar would otherwise fill train.log with \r frames.
            restore_print = _install_progress_bar(
                config.epochs, steps_per_epoch, real_stderr, handle
            )

        try:
            summary = finetune(
                data=data_path,
                model_dir=base,
                output_dir=str(out_dir),
                config=config,
                device=device,
            )
        except (ValueError, FileNotFoundError, TypeError) as err:
            print(
                "\ntrain_laya: training failed: %s" % err, file=sys.stderr, flush=True
            )
            if "model.safetensors" in str(err) or "Checkpoint directory" in str(err):
                print(
                    "  The base checkpoint has no PyTorch weights. backend/models/"
                    "laya-ml-checkpoint\n  holds only configs and the tokenizer -- the weights in "
                    "this project live only\n  in the ONNX graph. Use base = 'multilingual' to "
                    "download the real checkpoint.",
                    file=sys.stderr,
                    flush=True,
                )
            return 1
        finally:
            if restore_print is not None:
                restore_print()

        elapsed = time.time() - started
        _report(summary, out_dir, elapsed)
        _write_run_summary(summary, out_dir, config, elapsed, torch)
        return 0
    finally:
        sys.stdout, sys.stderr = real_stdout, real_stderr
        handle.close()
        print("log written to %s" % log_path, flush=True)


def _run_dry(laya_dry_run, data_path: str, base: str, config, gate: bool) -> tuple[int, dict[str, Any]]:
    """Run laya's dry run; when `gate`, refuse to continue if it dropped any item.

    Returns the exit code and the summary, whose train_items is what gives the
    progress bar an exact total.
    """
    try:
        summary = laya_dry_run(data=data_path, model_dir=base, config=config)
    except (ValueError, FileNotFoundError, TypeError) as err:
        print("train_laya: dry run failed: %s" % err, file=sys.stderr, flush=True)
        return 1, {}

    skipped = summary.get("skipped") or {}
    eval_skipped = summary.get("eval_skipped") or {}
    if not (skipped or eval_skipped):
        print(
            "dry-run: skipped {} -- every labelled question became a training item.",
            flush=True,
        )
    if gate and (skipped or eval_skipped):
        print(
            "\ntrain_laya: refusing to train. The dry run dropped items:\n"
            "  train: %r\n  eval:  %r\n"
            "A wrong label literal does not raise -- laya counts it under 'invalid_target'\n"
            "and drops it silently -- so this is the only thing between a format mistake\n"
            "and a wasted multi-hour run. Fix the corpus, or set allow_skipped = True."
            % (skipped, eval_skipped),
            file=sys.stderr,
            flush=True,
        )
        return 1, summary
    return 0, summary


def _report(summary: dict[str, Any], out_dir: Path, elapsed: float) -> None:
    report = summary.get("train_report") or {}

    print("\n=== Run ===", flush=True)
    print(
        "wall clock        %s" % time.strftime("%H:%M:%S", time.gmtime(elapsed)),
        flush=True,
    )
    print("train items       %d" % summary.get("train_items", 0), flush=True)
    print("calibration items %d" % summary.get("calibration_items", 0), flush=True)
    print(
        "eval items        %d (%s)"
        % (summary.get("eval_items", 0), report.get("eval_source") or "none"),
        flush=True,
    )
    print("skipped           %r" % (summary.get("skipped") or {},), flush=True)
    print(
        "epoch mean loss   %s"
        % [round(float(v), 4) for v in summary.get("epoch_loss") or []],
        flush=True,
    )

    mode = report.get("eval_mode")
    if mode and mode != "held_out":
        print(
            "\nWARNING: eval_mode is %r, not 'held_out'.\n  %s"
            % (mode, report.get("note")),
            flush=True,
        )

    comparison = report.get("comparison")
    if comparison:
        print("\n=== Overall deltas (after - before) ===", flush=True)
        for key in (
            "delta_accuracy",
            "delta_loss",
            "delta_ece",
            "delta_brier",
            "delta_brier_top1",
            "delta_mean_confidence",
        ):
            value = comparison.get(key)
            print(
                "%-24s %s" % (key, ("%+.4f" % value) if value is not None else "N/A"),
                flush=True,
            )
        if (comparison.get("delta_accuracy") or 0) <= 0:
            print(
                "\nWARNING: accuracy did not improve. Check the collapse warning above, and "
                "consider\n  more epochs, more corpus, or another seed before trusting this "
                "checkpoint.",
                flush=True,
            )

    _print_by_type(report)
    _print_env_block(summary, out_dir)

    print("\n=== Baseline to beat ===", flush=True)
    print(BASELINE_NOTE, flush=True)
    print(
        "\nRe-measure these against the fine-tuned graph before wiring the catalogue into\n"
        "intake (TRD stage 4c). Eval-split accuracy is an upper bound: the corpus is\n"
        "entirely synthetic and the model inherits its blind spots (TRD R9).",
        flush=True,
    )

    print("\n=== Artifacts in %s ===" % out_dir, flush=True)
    for name in (
        "model.safetensors",
        "rl_agent_config.json",
        "questions.json",
        "train_report.json",
        "run_summary.json",
        "train.log",
    ):
        path = out_dir / name
        if path.exists():
            print(
                "  %-24s %9.1f MB" % (name, path.stat().st_size / 1024**2), flush=True
            )
    for name in ("encoder", "tokenizer", "checkpoint_latest"):
        if (out_dir / name).is_dir():
            print("  %-24s (directory)" % name, flush=True)
    print(
        "\ncheckpoint_latest/ is a full checkpoint saved after every epoch. If the\n"
        "session dropped mid-run you can continue with base = '<out>/checkpoint_latest',\n"
        "though that restarts the LR schedule and re-draws the calibration split.",
        flush=True,
    )
    print("\nDownload %s from the Kaggle Output tab before the session expires." % out_dir, flush=True)


def _write_run_summary(
    summary: dict[str, Any], out_dir: Path, config, elapsed: float, torch
) -> None:
    """Everything needed to reconstruct the run, beside the checkpoint it produced."""
    from dataclasses import asdict

    try:
        import laya

        laya_version = getattr(laya, "__version__", "unknown")
    except Exception:
        laya_version = "unknown"

    payload = {
        "elapsed_seconds": round(elapsed, 1),
        "finished_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "torch_version": torch.__version__,
        "laya_version": laya_version,
        "device": device,
        "gpu": _describe_gpu() if torch.cuda.is_available() else None,
        "base": base,
        "config": asdict(config),
        "result": {k: v for k, v in summary.items() if k != "train_report"},
        "train_report": summary.get("train_report"),
    }
    path = out_dir / "run_summary.json"
    tmp = path.with_suffix(".json.tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, default=str)
    os.replace(tmp, path)

In [ ]:
main()